In [1]:
from rdflib import Graph,URIRef,Namespace,BNode,Literal
from rdflib.namespace import XSD,RDF
import pandas as pd
import argparse
from collections import defaultdict
from datetime import datetime
import json

In [2]:
graph = Graph()
print("Started loading graph...")
graph.parse('sample.ttl', format="turtle",publicID="https://example.org/")
print("Graph loaded successfully.")

Started loading graph...
Graph loaded successfully.


In [3]:
get_sensor_query = ''' 
PREFIX sosa: <http://www.w3.org/ns/sosa/>

SELECT DISTINCT ?sensor
WHERE {
  ?s sosa:madeBySensor ?sensor .
}
'''
sensor_set = set()
sensor_set.clear()
for sensor in graph.query(get_sensor_query):
    print(sensor.sensor)
    sensor_set.add(str(sensor[0]))

https://example.org/temp_sensor_1
https://example.org/humidity_sensor_1
https://example.org/temp_sensor_2
https://example.org/pressure_sensor_1
https://example.org/light_sensor_1
https://example.org/wind_sensor_1
https://example.org/precipitation_sensor_1
https://example.org/humidity_sensor_2
https://example.org/pressure_sensor_2
https://example.org/light_sensor_2
https://example.org/wind_sensor_2


In [8]:
prefix_tss = Namespace('https://w3id.org/tss#')
prefix_ex  = Namespace('http://example.org/')
prefix_sosa  = Namespace('http://www.w3.org/ns/sosa/')

In [20]:
final_graph = Graph()
final_graph.bind('tss',prefix_tss)
final_graph.bind('ex',prefix_ex)
final_graph.bind('sosa',prefix_sosa)


for sensor in sensor_set:
    tss_points = []
    test_query = f'''
    PREFIX sosa: <http://www.w3.org/ns/sosa/>

    SELECT ?READING ?TIME ?OBSERVATION ?observedProperty
    WHERE {{
        ?OBSERVATION a sosa:Observation ;
           sosa:resultTime ?TIME;
           sosa:hasSimpleResult ?READING;
           sosa:observedProperty ?observedProperty;
           sosa:madeBySensor <{sensor}>.

    }}

    ORDER BY ?TIME
    '''
    results = graph.query(test_query)
    for row in results:
        #print(sensor, ' value:' ,row.READING, ' time:', row.TIME, ' ID:' ,row.OBSERVATION, ' Observed property: ',row.observedProperty)

        
        data = {
            'time': row.TIME,
            'value': row.READING,
            'id': row.OBSERVATION,
            #'observedProperty' :  row.observedProperty
        }
        
        tss_points.append(data)
    json_object = json.dumps(tss_points) #serialize json object to a string
    
    #Create new graph 
    subject  = prefix_ex[f"snippet/{str(tss_points[0]['time'])[:9]}"] #this is the proper subject and should replace "URIRef(sensor)"
    #point_template = BNode(f'${URIRef(sensor)}') #blank node for the PointTemplate. the sensor uri reference is added here to make sure that temporary node has unique id
    #print(point_template) #for testing 
    temporary_node = BNode() #temporary node is created for each sensor at a time to ensure its uniqueness.

    final_graph.add((URIRef(sensor),RDF.type,prefix_tss.Snippet)) #temp
    final_graph.add((URIRef(sensor),prefix_tss.points,Literal(json_object, datatype=RDF.JSON))) #the json array with time, value, id.
    final_graph.add((URIRef(sensor),prefix_tss["from"],tss_points[0]['time'])) #from is a reserved word, hence worked around it this way
    final_graph.add((URIRef(sensor),prefix_tss.to,tss_points[-1]['time'])) #from is a reserved word, hence worked around it this way
    final_graph.add((URIRef(sensor),prefix_tss.pointType,prefix_sosa.Observation)) 

    #tss context json object
    context_obj = {
    "@context": {
        "id": "@id",
        "time": {
            "@id": "http://www.w3.org/ns/sosa/resultTime",
            "@type": "http://www.w3.org/2001/XMLSchema#dateTime"
        },
        "value": {
            "@id": "http://www.w3.org/ns/sosa/resultTime",
            "@type": "http://www.w3.org/2001/XMLSchema#integer"
        }
    }
}
    context_obj = json.dumps(context_obj) #serialize json object to a string
    
    final_graph.add((URIRef(sensor),prefix_tss.context,Literal(context_obj, datatype=RDF.JSON)))
    
    #temporary node part
    final_graph.add((URIRef(sensor),prefix_tss.about,temporary_node))
    final_graph.add((temporary_node, RDF.type, prefix_tss.PointTemplate))
    final_graph.add((temporary_node,prefix_sosa.madeBySensor,URIRef(sensor)))
    final_graph.add((temporary_node,prefix_sosa.observedProperty,row.observedProperty))


    '''
    print(json_object)
    print('start time: ', tss_points[0]['time']) #since readings are already sorted, first object holds start time
    print('end time: ', tss_points[-1]['time']) #while last object always holds last time
    print('-----------------------------------------------------------------------------------------------------------------------')
    '''

In [21]:
for subj, pred, obj in final_graph:
    print(subj, pred, obj)

https://example.org/temp_sensor_2 https://w3id.org/tss#about Na53ba46d3a5c4f488dce04ff395bf0db
N4b18f03345c048e58ec6a4d9a178b516 http://www.w3.org/ns/sosa/madeBySensor https://example.org/wind_sensor_1
https://example.org/temp_sensor_1 https://w3id.org/tss#from 2025-12-31T23:59:59+00:00
https://example.org/pressure_sensor_2 https://w3id.org/tss#from 2026-05-01T01:01:01+00:00
https://example.org/light_sensor_2 https://w3id.org/tss#about N006bc4e158e34a70b56e49b279517c58
https://example.org/wind_sensor_2 https://w3id.org/tss#from 2026-05-31T23:59:59+00:00
https://example.org/humidity_sensor_2 https://w3id.org/tss#about N532c8448145344e98d21319993595609
https://example.org/pressure_sensor_2 https://w3id.org/tss#context {"@context": {"id": "@id", "time": {"@id": "http://www.w3.org/ns/sosa/resultTime", "@type": "http://www.w3.org/2001/XMLSchema#dateTime"}, "value": {"@id": "http://www.w3.org/ns/sosa/resultTime", "@type": "http://www.w3.org/2001/XMLSchema#integer"}}}
Nac3b29a026f14a9b9d69ace